# Inheritance, Encapsulation and Polymorphism

Reuse behaviour, protect data and treat different objects the same way.

**You will learn to:**

- Extend a class with inheritance and `super()`
- Override methods
- Protect data with naming conventions and `@property`
- Use polymorphism and abstract base classes
- Choose between inheritance and composition

## 1. Inheritance

A **child** class inherits the attributes and methods of a **parent** class, and can add or change them.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} makes a sound"


class Dog(Animal):                       # Dog inherits from Animal
    def __init__(self, name, breed):
        super().__init__(name)           # run the parent's constructor
        self.breed = breed

    def fetch(self):
        return f"{self.name} fetches the ball"


rex = Dog("Rex", "Labrador")
print(rex.speak())       # inherited
print(rex.fetch())       # new in Dog
print(rex.breed)

### Override a method

In [ ]:
class Cat(Animal):
    def speak(self):                      # replaces the parent's version
        return f"{self.name} says Meow"


class Cow(Animal):
    def speak(self):
        base = super().speak()            # reuse the parent's version
        return base + " (moo)"


for animal in [Animal("Generic"), Cat("Tom"), Cow("Daisy")]:
    print(animal.speak())

In [ ]:
print(isinstance(rex, Dog), isinstance(rex, Animal))
print(issubclass(Dog, Animal))
print(Dog.__mro__)           # the lookup order Python follows

## 2. Encapsulation

**Encapsulation** keeps an object's data behind a controlled interface so it cannot be set to nonsense.

| Convention | Meaning |
|---|---|
| `name` | Public: use freely |
| `_name` | Internal: please do not touch from outside |
| `__name` | Name-mangled: harder to reach by accident |

Python does not enforce privacy. These are agreements between programmers.

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance           # internal

    @property
    def balance(self):                    # read access
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount

    def withdraw(self, amount):
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount


account = Account("Ada", 100)
account.deposit(50)
print(account.balance)

try:
    account.balance = 1_000_000
except AttributeError as error:
    print("Cannot set:", error)

try:
    account.withdraw(500)
except ValueError as error:
    print("Rejected:", error)

### A property with validation

In [ ]:
class Settings:
    def __init__(self, temperature=1.0):
        self.temperature = temperature        # goes through the setter

    @property
    def temperature(self):
        return self._temperature

    @temperature.setter
    def temperature(self, value):
        if not 0 <= value <= 1:
            raise ValueError("temperature must be between 0 and 1")
        self._temperature = value


settings = Settings(0.3)
settings.temperature = 0.8
print(settings.temperature)

try:
    settings.temperature = 5
except ValueError as error:
    print("Rejected:", error)

## 3. Polymorphism

**Polymorphism** means "many forms": different classes offer the same method, so calling code does not care which one it has.
This is how an app can switch between a real model and a test double.

In [ ]:
class ClaudeClient:
    def generate(self, prompt):
        return f"[Claude] answer to: {prompt}"


class MockClient:
    def generate(self, prompt):
        return "[Mock] fixed answer"


def run(client, prompt):
    return client.generate(prompt)          # works with ANY object that has generate()


for client in [ClaudeClient(), MockClient()]:
    print(run(client, "What is a token?"))

This style, where an object is used for what it can do rather than what it is, is called **duck typing**.

### Abstract base classes

An **abstract base class** (ABC) defines the methods every child must provide, and refuses to be created itself.

In [ ]:
from abc import ABC, abstractmethod


class LLMClient(ABC):
    @abstractmethod
    def generate(self, prompt: str) -> str:
        """Return the model's reply to `prompt`."""


class EchoClient(LLMClient):
    def generate(self, prompt):
        return f"echo: {prompt}"


class BrokenClient(LLMClient):
    pass                                     # forgot generate()


print(EchoClient().generate("hi"))

for cls in (LLMClient, BrokenClient):
    try:
        cls()
    except TypeError as error:
        print("Cannot create:", error)

### Special methods give objects built-in behaviour

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __add__(self, other):
        return Money(self.amount + other.amount)

    def __eq__(self, other):
        return self.amount == other.amount

    def __str__(self):
        return f"${self.amount:.2f}"


print(Money(5) + Money(2.5))
print(Money(1) == Money(1))

## 4. Inheritance or composition?

- Use **inheritance** for an "is a" relationship: a `Dog` *is an* `Animal`.
- Use **composition** for a "has a" relationship: a `Conversation` *has* messages, a `Car` *has an* engine.

Prefer composition when unsure. Deep inheritance trees are hard to follow and change.

In [ ]:
class Engine:
    def start(self):
        return "engine started"


class Car:
    def __init__(self):
        self.engine = Engine()               # composition

    def start(self):
        return self.engine.start()


print(Car().start())

## Try it

1. Create a `Shape` base class with an `area()` method that raises `NotImplementedError`. Add `Rectangle` and `Circle` children, then print the areas of a list of shapes in one loop.
2. Add a validated `price` property to a `Product` class that rejects negative numbers.
3. Turn `Shape` into an abstract base class and confirm you cannot create it directly.

In [ ]:
# Your answers here